# LuxMap — train lại model ON/OFF trên dataset Roboflow v4

1. Chia lại train / valid / test **theo video** (frame cùng video không bị tách sang hai tập).
2. Train YOLO11n và YOLO11s (640, cùng cấu hình), đánh giá trên test ở đúng ngưỡng backend (conf 0.25, IoU NMS 0.45, NMS theo lớp).
3. Xuất ONNX đúng dạng backend kiểm: input `[1,3,640,640]`, output `[1,6,N]`, `names = {0: 'normal', 1: 'out'}`, `nms=False`.

Cần: Runtime → GPU (T4); Colab Secret `ROBOFLOW_API_KEY` bật quyền cho notebook này.

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics==8.4.173 roboflow onnx onnxruntime

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
version = rf.workspace("dat-nguyen-tien-se181844").project("luxmap-street-light-detection").version(4)
try:
    ds = version.download("yolov11", location="/content/ds", overwrite=True)
except Exception as e:  # older format name, same YOLO folder layout
    print("yolov11 export unavailable, using yolov8:", e)
    ds = version.download("yolov8", location="/content/ds", overwrite=True)
print(ds.location)

## Chia lại theo nhóm nguồn (video / đợt chụp)

In [ ]:
import collections, csv, hashlib, random, re, shutil
from pathlib import Path
import yaml

SRC = Path("/content/ds")
DST = Path("/content/ds_grouped")
src_yaml = yaml.safe_load(open(SRC / "data.yaml"))
names = src_yaml["names"]
names = list(names.values()) if isinstance(names, dict) else list(names)
assert names == ["normal", "out"], names  # backend checks this exact order

items = []  # (image path, label path)
for split in ["train", "valid", "test"]:
    for img in sorted((SRC / split / "images").glob("*")):
        items.append((img, SRC / split / "labels" / (img.stem + ".txt")))
print("images:", len(items))

def original_stem(img):
    stem = img.name.split(".rf.")[0]
    return re.sub(r"_(jpe?g|png)$", "", stem, flags=re.I)

def counts(lbl):
    c = collections.Counter()
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            if line.strip():
                c[int(line.split()[0])] += 1
    return c

# 1) group key by source
key, kind, single_photos = {}, {}, []
for img, _ in items:
    s = original_stem(img)
    m = re.fullmatch(r"(IMG_\d+)_\d+", s)
    if m:
        key[img], kind[img] = "video:" + m.group(1), "video"
    elif re.fullmatch(r"IMG_\d+", s):
        single_photos.append((int(s[4:]), img))
        kind[img] = "photo"
    elif s.lower().startswith("screenshot"):
        key[img], kind[img] = "screenshot", "other"
    elif s.lower().startswith("whatsapp"):
        key[img], kind[img] = "whatsapp", "other"
    elif s.lower().startswith("suggested"):
        key[img], kind[img] = "universe", "other"
    else:
        key[img], kind[img] = "misc:" + s, "other"

# consecutive photo numbers = same shooting session -> one group
run_start, prev = None, None
for n, img in sorted(single_photos):
    if prev is None or n - prev > 1:
        run_start = n
    key[img] = f"photos:IMG_{run_start:04d}"
    prev = n

# 2) byte-identical images must share a group (union-find)
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
by_hash = {}
for img, _ in items:
    h = hashlib.md5(img.read_bytes()).hexdigest()
    if h in by_hash:
        parent[find(key[img])] = find(key[by_hash[h]])
    else:
        by_hash[h] = img
print("byte-identical duplicates:", len(items) - len(by_hash))

groups = collections.defaultdict(list)
for img, lbl in items:
    groups[find(key[img])].append((img, lbl))

def gstats(members):
    c = collections.Counter()
    for _, lbl in members:
        c.update(counts(lbl))
    return len(members), c[0], c[1]

# 3) valid + test come only from field video groups (the survey pipeline's domain); everything else trains
video_groups = sorted(g for g, ms in groups.items() if all(kind[i] == "video" for i, _ in ms))
other_groups = sorted(set(groups) - set(video_groups))
gs = {g: gstats(ms) for g, ms in groups.items()}
tot_img = sum(gs[g][0] for g in video_groups)
tot_out = sum(gs[g][2] for g in video_groups)
print(f"video groups: {len(video_groups)}, frames: {tot_img}, out boxes: {tot_out}")

TARGET = {"test": 0.20, "valid": 0.15}
best = None
rng = random.Random(0)
for _ in range(20000):
    order = video_groups[:]
    rng.shuffle(order)
    assign, acc = {}, {"test": [0, 0, 0], "valid": [0, 0, 0]}
    for g in order:
        n, nn, no = gs[g]
        for split in ["test", "valid"]:
            if acc[split][0] < TARGET[split] * tot_img:
                assign[g] = split
                acc[split] = [acc[split][0] + n, acc[split][1] + nn, acc[split][2] + no]
                break
        else:
            assign[g] = "train"
    if any(acc[s][1] == 0 or acc[s][2] == 0 for s in acc):
        continue  # each eval split needs both classes
    score = sum(abs(acc[s][0] / tot_img - TARGET[s]) + abs(acc[s][2] / max(tot_out, 1) - TARGET[s]) for s in acc)
    if best is None or score < best[0]:
        best = (score, assign)
assert best, "no split gives both classes to valid and test"
assign = best[1]
for g in other_groups:
    assign[g] = "train"

if DST.exists():
    shutil.rmtree(DST)
rows = []
for g, members in groups.items():
    split = assign[g]
    for img, lbl in members:
        (DST / split / "images").mkdir(parents=True, exist_ok=True)
        (DST / split / "labels").mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, DST / split / "images" / img.name)
        if lbl.exists():
            shutil.copy2(lbl, DST / split / "labels" / lbl.name)
        rows.append((img.name, original_stem(img), g, split))

with open("/content/split_manifest.csv", "w", newline="") as f:
    csv.writer(f).writerows([("export_file", "original_stem", "group", "split"), *sorted(rows, key=lambda r: (r[3], r[2], r[1]))])
yaml.safe_dump({"path": str(DST), "train": "train/images", "val": "valid/images", "test": "test/images", "names": dict(enumerate(names))},
               open(DST / "data.yaml", "w"), sort_keys=False)

summary = collections.defaultdict(lambda: [0, 0, 0, 0])
for g in groups:
    n, nn, no = gs[g]
    s = summary[assign[g]]
    s[0] += 1; s[1] += n; s[2] += nn; s[3] += no
for split in ["train", "valid", "test"]:
    g, n, nn, no = summary[split]
    print(f"{split:5s}: {g:3d} groups, {n:4d} images, normal boxes {nn:4d}, out boxes {no:4d}")
print("test groups:", sorted(g for g in assign if assign[g] == "test"))
print("valid groups:", sorted(g for g in assign if assign[g] == "valid"))

## Train YOLO11n và YOLO11s (cùng cấu hình)

In [ ]:
from ultralytics import YOLO

DATA = "/content/ds_grouped/data.yaml"
ARCHS = ["yolo11n", "yolo11s"]
for arch in ARCHS:
    YOLO(f"{arch}.pt").train(
        data=DATA, imgsz=640, epochs=200, patience=50, cos_lr=True, seed=0,
        flipud=0.0, fliplr=0.5, close_mosaic=15,
        project="/content/runs", name=arch, exist_ok=True,
    )

## Đánh giá trên test ở đúng điểm vận hành của backend

In [ ]:
import json, os, time
import numpy as np
from pathlib import Path

CONF, NMS_IOU, MATCH_IOU = 0.25, 0.45, 0.5  # backend: Ai:ConfidenceThreshold / Ai:IouThreshold
TEST = Path("/content/ds_grouped/test")

def box_iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def read_gt(lbl, w, h):
    out = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            p = line.split()
            if len(p) >= 5:
                c, x, y, bw, bh = int(p[0]), *map(float, p[1:5])
                out.append((c, [(x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h]))
    return out

def operating_point(weights):
    model = YOLO(weights)
    stats = {c: {"tp": 0, "fp": 0, "fn": 0} for c in model.names}
    out_seen_as_normal = 0
    for img in sorted((TEST / "images").glob("*")):
        r = model.predict(str(img), imgsz=640, conf=CONF, iou=NMS_IOU, verbose=False)[0]
        h, w = r.orig_shape
        gt = read_gt(TEST / "labels" / (img.stem + ".txt"), w, h)
        preds = [(int(c), b.tolist(), float(s)) for c, b, s in zip(r.boxes.cls, r.boxes.xyxy, r.boxes.conf)]
        for c in model.names:
            g = [b for k, b in gt if k == c]
            used = set()
            for _, pb, _ in sorted((p for p in preds if p[0] == c), key=lambda p: -p[2]):
                cand = [(box_iou(pb, gb), i) for i, gb in enumerate(g) if i not in used]
                v, i = max(cand, default=(0.0, -1))
                if v >= MATCH_IOU:
                    used.add(i); stats[c]["tp"] += 1
                else:
                    stats[c]["fp"] += 1
            stats[c]["fn"] += len(g) - len(used)
        for k, gb in gt:  # a dead lamp reported as lit is the costly mistake
            if k == 1 and any(p[0] == 0 and box_iou(p[1], gb) >= MATCH_IOU for p in preds):
                out_seen_as_normal += 1
    res = {}
    for c, s in stats.items():
        p = s["tp"] / (s["tp"] + s["fp"]) if s["tp"] + s["fp"] else None
        r_ = s["tp"] / (s["tp"] + s["fn"]) if s["tp"] + s["fn"] else None
        res[model.names[c]] = {**s, "precision": p, "recall": r_}
    res["out_detected_as_normal"] = out_seen_as_normal
    return res

report = {}
for arch in ARCHS:
    w = f"/content/runs/{arch}/weights/best.pt"
    m = YOLO(w).val(data=DATA, split="test", imgsz=640, plots=True, project="/content/runs", name=f"{arch}_test", exist_ok=True, verbose=False)
    report[arch] = {
        "test_mAP50": float(m.box.map50),
        "test_mAP50_95": float(m.box.map),
        "test_AP50_per_class": {m.names[int(c)]: float(ap) for c, ap in zip(m.ap_class_index, m.box.ap50)},
        "at_backend_threshold": operating_point(w),
    }
print(json.dumps(report, indent=2))

## Xuất ONNX, kiểm dạng backend nhận, đo độ trễ CPU

In [ ]:
import hashlib
import onnxruntime as ort

for arch in ARCHS:
    path = YOLO(f"/content/runs/{arch}/weights/best.pt").export(format="onnx", imgsz=640, nms=False, simplify=True, dynamic=False, batch=1)
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    inp, out = s.get_inputs()[0], s.get_outputs()[0]
    meta = s.get_modelmeta().custom_metadata_map
    assert inp.shape == [1, 3, 640, 640], inp.shape
    assert out.shape[:2] == [1, 6], out.shape
    assert meta["names"] == "{0: 'normal', 1: 'out'}", meta["names"]
    x = np.random.rand(1, 3, 640, 640).astype(np.float32)
    for _ in range(3):
        s.run(None, {inp.name: x})
    t = []
    for _ in range(20):
        t0 = time.perf_counter(); s.run(None, {inp.name: x}); t.append((time.perf_counter() - t0) * 1000)
    report[arch]["onnx"] = {
        "file": path, "sha256": hashlib.sha256(open(path, "rb").read()).hexdigest(),
        "bytes": os.path.getsize(path), "output_shape": out.shape, "colab_cpu_ms_median": float(np.median(t)),
    }
    print(arch, report[arch]["onnx"])

report["training"] = {"ultralytics": "8.4.173", "roboflow_version": 4, "imgsz": 640, "epochs": 200, "patience": 50, "seed": 0,
                      "split": "grouped by source video; valid/test from field video only"}
json.dump(report, open("/content/report.json", "w"), indent=2, default=str)

## Gói kết quả để tải về

In [ ]:
import shutil
pkg = Path("/content/luxmap_model_v4")
if pkg.exists():
    shutil.rmtree(pkg)
pkg.mkdir()
for arch in ARCHS:
    shutil.copy2(f"/content/runs/{arch}/weights/best.onnx", pkg / f"{arch}_best.onnx")
    shutil.copy2(f"/content/runs/{arch}/results.csv", pkg / f"{arch}_results.csv")
shutil.copy2("/content/report.json", pkg)
shutil.copy2("/content/split_manifest.csv", pkg)
shutil.make_archive("/content/luxmap_model_v4", "zip", pkg)
!ls -la /content/luxmap_model_v4.zip